# Face Autoencoder (CelebA) — Google Colab

A convolutional **autoencoder** (not a VAE — deterministic latent vector, reconstruction loss only) trained to reconstruct face images from [CelebA](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html).

**No manual dataset upload needed and no Kaggle account required.** Images are streamed from the [`nielsr/CelebA-faces`](https://huggingface.co/datasets/nielsr/CelebA-faces) mirror on the Hugging Face Hub using the `datasets` library.

> CelebA is released for **non-commercial research use only** — keep that in mind if you plan to use this beyond experimentation.

## Setup steps
1. Runtime → Change runtime type → select a GPU (T4 is fine) for faster training.
2. Run the cells below in order. The first run downloads a subset of images (~a couple minutes); later runs in the same session reuse the cached files.


## 1. Install dependencies

In [ ]:
!pip install -q datasets

## 2. Imports

In [ ]:
import os
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Download the dataset

Pulls images from the Hugging Face mirror of CelebA and saves them to disk as `.jpg` files.
Set `num_images=None` to download the full ~202k image dataset (slower, ~1.4GB); the default
below only grabs a few thousand so you can iterate quickly.


In [ ]:
FALLBACK_DIR = "/content/data/img_align_celeba"
NUM_FALLBACK_IMAGES = 5000  # set to None for the full ~202k image dataset


def download_celeba(directory: str = FALLBACK_DIR, num_images: int = NUM_FALLBACK_IMAGES):
    """Download CelebA face images from the Hugging Face Hub mirror.

    Skips the download if images are already present in `directory`.
    """
    from datasets import load_dataset

    os.makedirs(directory, exist_ok=True)

    existing = [f for f in os.listdir(directory) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
    if existing:
        print(f"Found {len(existing)} images already at {directory}, skipping download.")
        return directory

    print("Downloading CelebA faces from Hugging Face (nielsr/CelebA-faces)...")
    split = "train" if num_images is None else f"train[:{num_images}]"
    ds = load_dataset("nielsr/CelebA-faces", split=split)

    print(f"Saving {len(ds)} images to {directory} ...")
    for i, example in enumerate(ds):
        example["image"].save(os.path.join(directory, f"{i:06d}.jpg"))
        if (i + 1) % 1000 == 0:
            print(f"  saved {i + 1}/{len(ds)}")

    saved = [f for f in os.listdir(directory) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
    if not saved:
        raise RuntimeError("Download completed but no images were saved -- something went wrong.")

    print(f"Data ready at: {directory} ({len(saved)} images)")
    return directory


DATA_DIR = download_celeba()

## 4. Dataset class

In [ ]:
class CelebADataset(Dataset):
    """CelebA face image dataset.

    Args:
        directory: path to the folder of .jpg images.
        img_transform: torchvision transform to apply to each image.
        num_images: number of images to use. Pass -1 to use every image in `directory`.
    """

    def __init__(self, directory: str, img_transform=None, num_images: int = 2000):
        self.directory = directory
        self.img_transform = img_transform

        all_files = sorted(f for f in os.listdir(directory) if f.endswith(".jpg"))
        if not all_files:
            raise RuntimeError(f"No .jpg files found in {directory}")

        self.img_files = all_files if num_images == -1 else all_files[:num_images]

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.directory, self.img_files[idx])
        img = Image.open(img_path).convert("RGB")
        if self.img_transform:
            img = self.img_transform(img)
        return img, self.img_files[idx]

## 5. Model: Encoder / Decoder / Autoencoder\n\nPlain autoencoder — single deterministic latent vector, no KL term, no sampling.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, image_size: int = 128, embed_dim: int = 128):
        super().__init__()
        self.feature_size = image_size // 8  # 3 stride-2 convs halve dims each time
        self.conv = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(256 * self.feature_size * self.feature_size, embed_dim)

    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        return self.fc(x)


class Decoder(nn.Module):
    def __init__(self, image_size: int = 128, embed_dim: int = 128):
        super().__init__()
        self.feature_size = image_size // 8
        self.fc = nn.Linear(embed_dim, 256 * self.feature_size * self.feature_size)
        self.unflatten = nn.Unflatten(1, (256, self.feature_size, self.feature_size))
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = self.unflatten(x)
        return self.deconv(x)


class Autoencoder(nn.Module):
    def __init__(self, image_size: int = 128, embed_dim: int = 128):
        super().__init__()
        self.encoder = Encoder(image_size, embed_dim)
        self.decoder = Decoder(image_size, embed_dim)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

## 6. Helper: show side-by-side comparison images

In [ ]:
def show_comparison(original, reconstructed, img_name, epoch):
    orig_arr = original.detach().cpu().permute(1, 2, 0).numpy()
    recon_arr = reconstructed.detach().cpu().permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(orig_arr)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(recon_arr)
    axes[1].set_title("Reconstructed")
    axes[1].axis("off")

    plt.suptitle(f"Epoch {epoch} — {img_name}")
    plt.tight_layout()
    plt.show()

## 7. Training loop

Batched with `DataLoader`, with checkpointing so you can resume training later in the same
or a future Colab session. Mount Google Drive first if you want checkpoints to survive
runtime resets (`from google.colab import drive; drive.mount('/content/drive')`), then point
`checkpoint_path` at a path under `/content/drive/MyDrive/...`.


In [ ]:
def train(
    dataset,
    model,
    epochs=20,
    batch_size=32,
    lr=1e-3,
    display_interval=5,
    checkpoint_path="autoencoder_checkpoint.pth",
    resume=False,
):
    model = model.to(device)
    loss_fn = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    start_epoch = 0
    loss_history = []

    if resume and os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        loss_history = checkpoint.get("loss_history", [])
        print(f"Resuming from epoch {start_epoch}")

    if start_epoch >= epochs:
        print(f"Checkpoint already at epoch {start_epoch} >= target {epochs}. Nothing to do.")
        return model, loss_history

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

    random.seed(10)
    sample_idx = random.randint(0, len(dataset) - 1)
    sample_img, sample_name = dataset[sample_idx]
    sample_img = sample_img.unsqueeze(0).to(device)

    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss = 0.0

        for imgs, _ in tqdm(loader, desc=f"Epoch {epoch + 1}/{epochs}"):
            imgs = imgs.to(device)

            optimizer.zero_grad()
            reconstructions = model(imgs)
            loss = loss_fn(reconstructions, imgs)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)

        avg_loss = running_loss / len(dataset)
        loss_history.append(avg_loss)
        print(f"Epoch [{epoch + 1}/{epochs}] avg loss: {avg_loss:.4f}")

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": avg_loss,
            "loss_history": loss_history,
        }, checkpoint_path)

        if (epoch + 1) % display_interval == 0 or (epoch + 1) == epochs:
            model.eval()
            with torch.no_grad():
                reconstructed = model(sample_img)
            show_comparison(sample_img.squeeze(0), reconstructed.squeeze(0), sample_name, epoch + 1)

    return model, loss_history


def plot_loss(loss_history):
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Avg reconstruction loss (MSE)")
    plt.title("Training loss over epochs")
    plt.grid(True, alpha=0.3)
    plt.show()

## 8. Run it

Adjust `IMAGE_SIZE`, `NUM_IMAGES`, `BATCH_SIZE`, and `EPOCHS` as needed. Set `NUM_IMAGES = -1`
to use every image already downloaded to `DATA_DIR`.


In [ ]:
IMAGE_SIZE = 128
EMBED_DIM = 128
NUM_IMAGES = 2000       # -1 to use every downloaded image
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3
DISPLAY_INTERVAL = 1
CHECKPOINT_PATH = "autoencoder_checkpoint.pth"

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])
dataset = CelebADataset(DATA_DIR, img_transform=transform, num_images=NUM_IMAGES)
print(f"Dataset ready: {len(dataset)} images at {IMAGE_SIZE}x{IMAGE_SIZE}")

model = Autoencoder(image_size=IMAGE_SIZE, embed_dim=EMBED_DIM)

model, loss_history = train(
    dataset,
    model,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    display_interval=DISPLAY_INTERVAL,
    checkpoint_path=CHECKPOINT_PATH,
    resume=False,
)

In [ ]:
plot_loss(loss_history)

## 9. Resume training later

Re-run this cell with a higher `EPOCHS` value and `resume=True` to continue from the last
checkpoint (e.g. after your Colab runtime disconnects), as long as `CHECKPOINT_PATH` still
points to a file that exists (use a Google Drive path if you want it to survive a runtime reset).


In [ ]:
EPOCHS = 40  # increase this to train further

model, loss_history = train(
    dataset,
    model,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    display_interval=DISPLAY_INTERVAL,
    checkpoint_path=CHECKPOINT_PATH,
    resume=True,
)

In [ ]:
plot_loss(loss_history)